# Gold: Analyseklare tabeller

Bygger Gold-tabeller fra Silver-data med ren SQL.

**Forutsetning:** `02_silver` er kjørt.

Fem tabeller:

1. **pensjonsandel_trend** — vektet landsgjennomsnitt per år
2. **top_kommuner_pensjonsalder** — topp 20 kommuner med høyest andel 55+
3. **naering_pensjonsvolum** — næringer rangert etter pensjonsvolum
4. **aldersgruppe_fordeling** — fordeling siste år
5. **aldersgruppe_trend** — andeler per aldersgruppe over tid

In [0]:
CREATE SCHEMA IF NOT EXISTS pensjon_lakehouse.gold

## pensjonsandel_trend

Vektet nasjonalt gjennomsnitt: `SUM(55+) / SUM(total)`, ikke `AVG(kommuneandeler)`.

In [0]:
CREATE OR REPLACE TABLE pensjon_lakehouse.gold.pensjonsandel_trend AS
SELECT
    year,
    SUM(pension_age_befolkning) AS total_55_pluss,
    SUM(total_befolkning)       AS total_befolkning,
    ROUND(
        CAST(SUM(pension_age_befolkning) AS DOUBLE)
        / NULLIF(SUM(total_befolkning), 0)
        * 100,
        2
    ) AS pensjonsandel_pst
FROM pensjon_lakehouse.silver.befolkning_pensjon
GROUP BY year
ORDER BY year

In [0]:
SELECT * FROM pensjon_lakehouse.gold.pensjonsandel_trend

## top_kommuner_pensjonsalder

In [0]:
CREATE OR REPLACE TABLE pensjon_lakehouse.gold.top_kommuner_pensjonsalder AS
SELECT
    kommune_code,
    kommune_label,
    year,
    total_befolkning,
    pension_age_befolkning,
    pension_age_share
FROM pensjon_lakehouse.silver.befolkning_pensjon
WHERE year = (SELECT MAX(year) FROM pensjon_lakehouse.silver.befolkning_pensjon)
ORDER BY pension_age_share DESC
LIMIT 20

In [0]:
SELECT * FROM pensjon_lakehouse.gold.top_kommuner_pensjonsalder

## naering_pensjonsvolum

In [0]:
CREATE OR REPLACE TABLE pensjon_lakehouse.gold.naering_pensjonsvolum AS
SELECT
    naering_code,
    naering_label,
    kvartal,
    lonsstakere,
    manedslonn,
    estimert_pensjonsvolum
FROM pensjon_lakehouse.silver.naering_pensjon
WHERE kvartal = (SELECT MAX(kvartal) FROM pensjon_lakehouse.silver.naering_pensjon)
  AND estimert_pensjonsvolum IS NOT NULL
ORDER BY estimert_pensjonsvolum DESC

In [0]:
SELECT * FROM pensjon_lakehouse.gold.naering_pensjonsvolum

## aldersgruppe_fordeling

In [0]:
CREATE OR REPLACE TABLE pensjon_lakehouse.gold.aldersgruppe_fordeling AS
SELECT
    aldersgruppe,
    aldersgruppe_sortering,
    SUM(befolkning) AS befolkning,
    ROUND(
        CAST(SUM(befolkning) AS DOUBLE)
        / NULLIF(SUM(SUM(befolkning)) OVER (), 0),
        4
    ) AS andel
FROM pensjon_lakehouse.silver.befolkning_aldersgrupper
WHERE year = (SELECT MAX(year) FROM pensjon_lakehouse.silver.befolkning_aldersgrupper)
GROUP BY aldersgruppe, aldersgruppe_sortering
ORDER BY aldersgruppe_sortering

In [0]:
SELECT * FROM pensjon_lakehouse.gold.aldersgruppe_fordeling

## aldersgruppe_trend

In [0]:
CREATE OR REPLACE TABLE pensjon_lakehouse.gold.aldersgruppe_trend AS
SELECT
    year,
    aldersgruppe,
    aldersgruppe_sortering,
    SUM(befolkning) AS befolkning,
    ROUND(
        CAST(SUM(befolkning) AS DOUBLE)
        / NULLIF(SUM(SUM(befolkning)) OVER (PARTITION BY year), 0),
        4
    ) AS andel
FROM pensjon_lakehouse.silver.befolkning_aldersgrupper
GROUP BY year, aldersgruppe, aldersgruppe_sortering
ORDER BY year, aldersgruppe_sortering

In [0]:
SELECT * FROM pensjon_lakehouse.gold.aldersgruppe_trend

## Ferdig

Alle Gold-tabeller ligger nå i `pensjon_lakehouse.gold.*`.

Sett opp dashboardet med **04_dashboard_setup**.